# Document Loader Evaluation for Retrieval-Augmented Generation (RAG)

## Introduction
This notebook presents an exploratory comparison of multiple document loaders commonly used in RAG pipelines. The objective is to examine how different extraction approaches perform on the **same input document** while keeping the downstream workflow unchanged.

> **This is not a definitive benchmark.** The observations recorded throughout this notebook are specific to the document used during testing and should not be interpreted as universal rankings, i am open to hearing suggestions.
Email: m.shaheer1915@gmail.com

## Purpose
- Compare multiple PDF document loaders.
- Observe extraction quality on challenging documents.
- Discuss how extraction quality affects downstream chunking and retrieval.
- Identify strengths and weaknesses of OCR and non-OCR approaches.

## About the Test Document
**Original Document:** *https://drive.google.com/file/d/1f0oKyutfw_lPKhlGeWJSkIX9egCza_oF/view?usp=sharing*

The original experiments were performed using an older digitized book containing a mixture of selectable text, scanned image pages, inconsistent formatting, and sections that behave entirely like embedded images. These characteristics significantly influence extraction quality.

If your document is a clean digital PDF with selectable text, a lightweight loader such as PyPDFLoader may be entirely sufficient. This notebook instead focuses on more challenging, mixed-format and unstructured documents where OCR-assisted approaches become increasingly valuable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
#Enter your own document path or the path to the document used for my test (drive link above)
pdf_path = ""

In [ ]:
!pip install langchain_community

In [ ]:
!pip install langchain_core

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 10.6 MB/s eta 0:00:00


# Test 1 – PyPDFLoader

## Objective
Evaluate the baseline text extraction quality using PyPDFLoader. This loader extracts embedded PDF text without OCR and serves as a useful baseline for comparison.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
def extract_documents():
    """
    Load PDF and convert into LangChain Documents.
    """
    loader = PyPDFLoader(pdf_path)

    pages = loader.load()

    documents = []

    for page in pages:
        text = page.page_content.strip()

        if not text:
            continue

        metadata = {
            "source": "22 Cells in Nuremberg",
            "page": page.metadata.get("page", 0)
        }

        documents.append(
            Document(
                page_content=text,
                metadata=metadata
            )
        )

    return documents

pypdf_docs = extract_documents()

KeyboardInterrupt: 

In [ ]:
for doc in pypdf_docs[10:40]:
    print(f"--- Page {doc.metadata['page']} ---")
    print(doc.page_content)
    print()

--- Page 13 ---
ei roots of primitive loyalties and hatreds requiring o 
so | 
“ONE! ae te 
"PAN-GERMANISM AND | 
IDEOLOGY 
be fostered perhaps more soe iden ‘cultu 
The seeds of PrOprens are _always ean in ev 
‘sunlight and showers. Also present in So “pale 
sunlight and showers. Also present in every cul 
Germany into neo-paganism and barbarism simply 
ing a heavy dressing of hate-rousing propaganda — 
makeup. Let me illustrate with three quotations: 
First: “It is necessary that our Spo ee 
_ the death cries of men without number.” — 
Second: “Frankly, we are and must be. barbarians . 
countries ... They call us barbarians. What of it? We scor 
them and their abuse...Our troops must achieve victo 
What else matters?” ea 
Third: “The new Europe will be a continent resto 
barbarism... And this time the foundation for the new ex 
rope will be laid, not by priests and diplomats, but b 
pirates of destiny ... Now, at last, we may frankly cont 
that the Gospel has lost all meaning for us.

## Observations
PyPDFLoader performs well on digitally generated PDFs, but its performance degrades when pages contain scanned images or inconsistent formatting. Because this document mixes selectable text with image-based content, several sections are fragmented or omitted, reducing the quality of text available for embedding.

In [ ]:
!pip install unstructured[pdf]

In [ ]:
!pip install langchain-community unstructured

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [ ]:
from langchain_community.document_loaders import UnstructuredPDFLoader
def unstructured_extract(pdf_path):
    loader = UnstructuredPDFLoader(
        pdf_path,
        mode="elements",
        strategy="fast"
    )
    docs = loader.load()
    return docs

Unstructed_docs = unstructured_extract(pdf_path)

In [ ]:
print(Unstructed_docs)

[Document(metadata={'source': '/content/drive/MyDrive/22-Cells-in-Nuremberg.pdf', 'coordinates': {'points': ((233.2, 23.968000000044356), (233.2, 37.00894444444447), (248.97827555552274, 37.00894444444447), (248.97827555552274, 23.968000000044356)), 'system': 'PixelSpace', 'layout_width': 289.8, 'layout_height': 486.2}, 'file_directory': '/content/drive/MyDrive', 'filename': '22-Cells-in-Nuremberg.pdf', 'last_modified': '2026-04-18T09:29:40', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/pdf', 'category': 'Header', 'element_id': 'de25c32323b7e43d3b49e066c8cf10a0'}, page_content='\\ |'), Document(metadata={'source': '/content/drive/MyDrive/22-Cells-in-Nuremberg.pdf', 'coordinates': {'points': ((31.0, 36.73225000000218), (31.0, 41.20447222222219), (50.20528749999046, 41.20447222222219), (50.20528749999046, 36.73225000000218)), 'system': 'PixelSpace', 'layout_width': 289.8, 'layout_height': 486.2}, 'file_directory': '/content/drive/MyDrive', 'filename': '22-Cells-in-Nur

In [ ]:
print(len(unstructured_docs))

59


## Inspecting the Output
The raw output contains many small document objects, making qualitative evaluation difficult. Before comparing loaders, it is helpful to inspect the extracted content in a more readable form.

In [ ]:
for doc in Unstructed_docs[10:40]:
    print(doc.page_content)
    print()

DEDICATION

TO JUNE AND DOC

THIS BOOK IS THE COMPLETE TEXT OF THE HARDCOVER BOOK

A MACFADDEN BOOK .. .

1961

COPYRIGHT 1947 BY DOUGLAS M. KELLEY ALL RIGHTS RESERVED PUBLISHED BY ARRANGEMENT WITH CHILTON COMPANY, PUBLISHERS

MacFadden Books Are Published by MacFadden Publications, Inc. 205 East 42nd Street, New York 17, New York

PRINTED IN U.S.A.

ke

ag

senna

o

CONTENTS

INTRODUCTION

Part One

THE ENVIRONMENT

PAN-GERMANISM AND NAZI IDEOLOGY: NUREMBERG JAIL

Part Two

THE POLICY MAKERS

RUDOLF HESS ™ ALFRED ROSENBERG HERMANN GOERING

Part Three

THE SALESMEN

HANS FRITZSCHE

BALDUR VON SCHIRACH

JOACHIM VON RIBBENTROP CONSTANTIN VON NEURATH FRANZ VON PAPEN

0 ONO

12

15

21

35



## Merging Adjacent Elements
The following helper function merges neighbouring elements into larger chunks purely for readability during evaluation. This does not change the extraction itself.

In [ ]:
def merge_documents(docs, min_length=200):
    merged_docs = []
    buffer = ""

    for doc in docs:
        text = doc.page_content.strip()

        if not text:
            continue

        buffer += " " + text

        if len(buffer) >= min_length:
            doc.page_content = buffer.strip()
            merged_docs.append(doc)
            buffer = ""

    # leftover
    if buffer:
        doc.page_content = buffer.strip()
        merged_docs.append(doc)

    return merged_docs


merged_docs = merge_documents(Unstructed_docs)

In [ ]:
for doc in merged_docs:
    print(doc.page_content)
    print()

\ | RUE, .OUGLAS M KELLEY, M.D. A ROGUE'S GALLERY OF THE ARCH CRIMINALS | OF ALL TIME BY THE OFFICIAL UNITED STATES — PSYCHIATRIST WHO EXAMINED THEM AND LEARNED THEIR MOST INTIMATE SECRETS. BT aN —Lewis M. Terman Emeritus Professor of Psychology Stanford University

i 22 CELLS IN "NUREMBERG by Douglas M. Kelley, M.D. Psychiatrist to the Nuremberg Jail A MACFADDEN BOOK DEDICATION TO JUNE AND DOC THIS BOOK IS THE COMPLETE TEXT OF THE HARDCOVER BOOK A MACFADDEN BOOK .. .

1961 COPYRIGHT 1947 BY DOUGLAS M. KELLEY ALL RIGHTS RESERVED PUBLISHED BY ARRANGEMENT WITH CHILTON COMPANY, PUBLISHERS MacFadden Books Are Published by MacFadden Publications, Inc. 205 East 42nd Street, New York 17, New York

PRINTED IN U.S.A. ke ag senna o CONTENTS INTRODUCTION Part One THE ENVIRONMENT PAN-GERMANISM AND NAZI IDEOLOGY: NUREMBERG JAIL Part Two THE POLICY MAKERS RUDOLF HESS ™ ALFRED ROSENBERG HERMANN GOERING

Part Three THE SALESMEN HANS FRITZSCHE BALDUR VON SCHIRACH JOACHIM VON RIBBENTROP CONSTANTIN VON N

## Analysis
Compared with the baseline loader, the Unstructured loader preserves semantic grouping more effectively and produces cleaner chunks. Nevertheless, the quality is still fundamentally limited by the source document.

# OCR-Based Extraction
Traditional PDF parsers cannot recover text from image-only pages. The following experiments introduce OCR to determine whether recognition improves extraction quality for complex documents.

In [ ]:
!pip install pytesseract
!pip install pdf2image

In [ ]:
!apt-get install -y poppler-utils
!apt-get install -y tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
!apt-get install -y libmagic-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  libmagic-dev
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 105 kB of archives.
After this operation, 389 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libmagic-dev amd64 1:5.41-3ubuntu0.1 [105 kB]
Fetched 105 kB in 1s (209 kB/s)
Selecting previously unselected package libmagic-dev:amd64.
(Reading database ... 118224 files and directories currently installed.)
Preparing to unpack .../libmagic-dev_1%3a5.41-3ubuntu0.1_amd64.deb ...
Unpacking libmagic-dev:amd64 (1:5.41-3ubuntu0.1) ...
Setting up libmagic-dev:amd64 (1:5.41-3ubuntu0.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import pytesseract
from pdf2image import convert_from_path

def ocr_docs(pdf_path, start_page=30, end_page=40):
    """
    Extract text using OCR from a specific page range.

    Args:
        pdf_path (str): Path to PDF
        start_page (int): starting page (1-based index)
        end_page (int): ending page (inclusive)

    Returns:
        List of extracted text (one per page)
    """

    images = convert_from_path(
        pdf_path,
        first_page=start_page,
        last_page=end_page
    )

    texts = []
    for i, img in enumerate(images):
        print(f"OCR processing page {start_page + i}...")
        text = pytesseract.image_to_string(img)
        texts.append(text)

    return texts


ocr_docs = ocr_docs(pdf_path, start_page=30, end_page=35)

OCR processing page 30...
OCR processing page 31...
OCR processing page 32...
OCR processing page 33...
OCR processing page 34...
OCR processing page 35...


In [ ]:
print(ocr_docs)

[') gn\n\nomewhat incongruous flying bo\nlined, and made of soft leather with two\n\nlent him a military bearing | that\n_ have given. :\nPsychiatrically, he was Med and responsive.\nwas reserved and his general attitude formal, bi\n‘the impression of making a real attempt at c\nHis stream of thought was curtailed as a result\n__nesia, the majority of his responses being, “I do’\nor “I cannot remember.” He claimed to be\nmember his birth date, birthplace, date of leaving\nor any fact or detail whatsoever of his early life.\nThe next morning when I examined him again, he clai\nhe could not remember anything that had taken place\nhis imprisonment in England, and only vaguely rem\nthe plane trip to Nuremberg across the Channel. Hi\nremember very little of the details of his admis\n_ prison, but did recall his parcels, and again asked\n- surance that they were in a safe place where they cou! d\nbe tampered with.\nAt this time Hess’s mood was somewhat depressed,\nshowed generally normal rea

## Observations
Raw OCR successfully recovers text from image-based regions that conventional loaders miss. However, OCR errors, incorrect spacing and recognition mistakes remain visible, particularly on degraded pages.

In [ ]:
from unstructured.partition.pdf import partition_pdf

def unstructured_extract(pdf_path):
    print("Partitioning documents")

    # Load everything first
    elements = partition_pdf(
        filename=pdf_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["image"],
        extract_image_block_to_payload=True
    )

    # Keep only page 30 to 33
    filtered_elements = [
        el for el in elements
        if hasattr(el, "metadata")
        and hasattr(el.metadata, "page_number")
        and 30 <= el.metadata.page_number <= 33
    ]

    print(f"Extracted {len(filtered_elements)} elements from pages 30 to 33")
    return filtered_elements


unstructured_docs = unstructured_extract(pdf_path)

Partitioning documents


yolox_l0.05.onnx:   0%|          | 0.00/217M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

Extracted 59 elements from pages 30 to 33


In [ ]:
print(unstructured_docs)

[<unstructured.documents.elements.Text object at 0x78737cdf0d40>, <unstructured.documents.elements.NarrativeText object at 0x78737cddf9b0>, <unstructured.documents.elements.Text object at 0x78737cdf0140>, <unstructured.documents.elements.NarrativeText object at 0x78737cdf02c0>, <unstructured.documents.elements.Text object at 0x78737cdf0ec0>, <unstructured.documents.elements.Text object at 0x78737cdf0440>, <unstructured.documents.elements.NarrativeText object at 0x78737cdf05c0>, <unstructured.documents.elements.Text object at 0x78737cdf1040>, <unstructured.documents.elements.NarrativeText object at 0x78737cdf0740>, <unstructured.documents.elements.Text object at 0x78737cdf08c0>, <unstructured.documents.elements.NarrativeText object at 0x78737cdf0a40>, <unstructured.documents.elements.Text object at 0x78737cdf0bc0>, <unstructured.documents.elements.Text object at 0x78737cdf1340>, <unstructured.documents.elements.Text object at 0x78737cdf11c0>, <unstructured.documents.elements.Text object

In [ ]:
print(set([str(type(doc)) for doc in unstructured_docs]))

{"<class 'unstructured.documents.elements.NarrativeText'>", "<class 'unstructured.documents.elements.Text'>", "<class 'unstructured.documents.elements.Header'>"}


In [ ]:
print(unstructured_docs[30].to_dict())

{'type': 'UncategorizedText', 'element_id': '5f58798e695cdd48efa8bbcf34f31db8', 'text': 'In Jess technical terms, Rudolf Hess was an introvel', 'metadata': {'coordinates': {'points': ((np.float64(210.9722222222222), np.float64(924.7850694444282)), (np.float64(210.9722222222222), np.float64(964.4841435185186)), (np.float64(1305.7501284722855), np.float64(964.4841435185186)), (np.float64(1305.7501284722855), np.float64(924.7850694444282))), 'system': 'PixelSpace', 'layout_width': 1380, 'layout_height': 2372}, 'last_modified': '2026-04-18T09:29:40', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 32, 'file_directory': '/content/drive/MyDrive', 'filename': '22-Cells-in-Nuremberg.pdf', 'parent_id': '382c319cea3b39256b6c33cddce108a1'}}


In [ ]:
for doc in unstructured_docs:
  print(doc.to_dict()["text"])

:
: aoe incongruous. Hee ares g lined, and made of soft leather with two zipper lent him a military Dearing | that no gene of ‘ins. cae given.
- Psychiatrically, he was alert one responsive. I
was reserved and his general attitude formal, b ‘the impression of making a real attempt at His stream of thought was curtailed as a result ro} ‘nesia, the majority of his responses being, “I do not kn or “I cannot remember.” He claimed to be unable - member his birth date, birthplace, date of leaving G or any fact or detail whatsoever of his early life.
Se ae ae
The next morning when I examined him again, he clai
he could not remember anything that had taken plac his imprisonment in England, and only vaguely reme =<" the plane trip to Nuremberg across the Channel. H remember very little of the details of his admission prison, but did recall his parcels, and again asked for surance that they were in a safe ace eee oe could be tampered with.
ell
showed “generally normal reactions. In every way, fo

## Analysis
Combining OCR with Unstructured improves document coverage, but noisy OCR output can still propagate through the pipeline. Results depend heavily on scan quality and document layout.

In [ ]:
!pip install marker-pdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796

In [ ]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from langchain_core.documents import Document
import re

converter = PdfConverter(
    artifact_dict=create_model_dict(),
    config={
        "force_ocr": True,
        "output_format": "markdown",
        "paginate_output": True,
        "page_range": list(range(29,33)),
    }
)

rendered = converter(pdf_path)
text, _, images = text_from_rendered(rendered)

pages = re.split(r'\n\n\d+\s*-{48}\n\n', text)




























































































































Recognizing Text: 100%|██████████| 88/88 [01:45<00:00,  1.20s/it]
Detecting bboxes: 0it [00:00, ?it/s]


In [ ]:
for i, page_text in enumerate(pages):
    page_text = page_text.strip()
    if not page_text:
        continue
    print(f"--- Page {30 + i} ---")
    print(page_text)
    print()

--- Page 30 ---
{29}------------------------------------------------

On arrival at the jail, Hess was, physically, in good shape, but thin. He was dressed in his Luftwaffe uniform, though without insignia. However, his heel-clicking stiffness and somewhat incongruous flying boots—high and black, thickly lined, and made of soft leather with two zippers on each—lent him a military bearing that no array of insignia could

Psychiatrically, he was alert and responsive. His approach was reserved and his general attitude formal, but he gave the impression of making a real attempt at cooperation. His stream of thought was curtailed as a result of his amnesia, the majority of his responses being, "I do not know," or "I cannot remember." He claimed to be unable to remember his birth date, birthplace, date of leaving Germany,

or any fact or detail whatsoever of his early life.

The next morning when I examined him again, he claimed he could not remember anything that had taken place during his 

# Final Comparison
Marker produced the strongest overall extraction for this challenging document by combining OCR with layout-aware reconstruction. While computationally more expensive, it generated cleaner reading order and better preserved document structure.

| Loader | Clean PDFs | Mixed/Unstructured PDFs | OCR | Overall |
|---|---|---|---|---|
| PyPDFLoader | Good | Poor | No | Best for clean digital PDFs |
| UnstructuredPDFLoader | Excelent | Moderate | Optional | Better semantic grouping |
| Raw OCR | Fair | Good | Yes | Recovers scanned text but noisy |
| Unstructured + OCR | Good | Good | Yes | Better coverage with some noise |
| Marker PDF | Excellent | Excellent | Yes | Best overall for difficult documents |

The optimal loader depends on the document being processed rather than a universally best solution.